In [1]:
import pandas as pd

# Load clean CSV files
batting = pd.read_csv("batting_summary_clean.csv")
bowling = pd.read_csv("bowling_summary_clean.csv")
players = pd.read_csv("player_info_clean.csv")

print("Batting:", batting.shape)
print("Bowling:", bowling.shape)
print("Players:", players.shape)


Batting: (852, 13)
Bowling: (595, 13)
Players: (304, 12)


In [2]:
batting_summary = batting.groupby("player_id").agg(
    player_name=("player_name", "first"),
    team=("team", "first"),

    total_runs=("runs", "sum"),
    balls_faced=("balls", "sum"),
    fours=("fours", "sum"),
    sixes=("sixes", "sum"),
    innings_batted=("match_name", "nunique")
).reset_index()
batting_summary

,player_id,player_name,team,total_runs,balls_faced,fours,sixes,innings_batted
0,P1,Rohit Sharma (c),India,257,164,24,15,8
1,P100,Chris Greaves,Scotland,13,13,0,0,2
2,P106,Kashyap,Oman,16,33,1,1,3
3,P107,Shoaib Khan,Oman,11,27,1,0,2
4,P108,Mehran Khan,Oman,47,36,4,2,4
...,...,...,...,...,...,...,...,...
242,P92,George Munsey,Scotland,124,89,8,9,4
243,P94,Richie Berrington (c),Scotland,102,73,4,5,3
244,P95,Michael Leask,Scotland,40,25,0,4,2
245,P96,Brandon McMullen,Scotland,140,82,13,8,3


In [3]:
# Strike Rate
batting_summary["strike_rate"] = (
    batting_summary["total_runs"] / batting_summary["balls_faced"]
) * 100

# Boundary Runs
batting_summary["boundary_runs"] = (
    batting_summary["fours"] * 4 +
    batting_summary["sixes"] * 6
)

# Boundary %
batting_summary["boundary_pct"] = (
    batting_summary["boundary_runs"] / batting_summary["total_runs"]
) * 100


In [4]:
def overs_to_balls(overs):
    whole_overs = int(overs)
    balls = round((overs - whole_overs) * 10)
    return whole_overs * 6 + balls


# Convert overs column into balls
bowling["balls"] = bowling["overs"].apply(overs_to_balls)

# Aggregate bowling stats
bowling_summary = bowling.groupby("player_id").agg(
    player_name=("player_name", "first"),
    team=("team", "first"),
    total_wickets=("wickets", "sum"),
    runs_conceded=("runs", "sum"),
    balls_bowled=("balls", "sum"),
    innings_bowled=("match_name", "nunique")
).reset_index()


In [5]:
# Overs bowled back
bowling_summary["overs_bowled"] = bowling_summary["balls_bowled"] / 6

# Economy
bowling_summary["economy"] = (
    bowling_summary["runs_conceded"] / bowling_summary["overs_bowled"]
)
bowling_summary.head()

,player_id,player_name,team,total_wickets,runs_conceded,balls_bowled,innings_bowled,overs_bowled,economy
0,P100,Chris Greaves,SCOTLAND,2,30,30,3,5.000000,6.000000
1,P101,Safyaan Sharif,SCOTLAND,4,82,48,2,8.000000,10.250000
2,P102,Christopher Sole,SCOTLAND,2,97,60,3,10.000000,9.700000
3,P103,Mark Watt,SCOTLAND,3,98,72,3,12.000000,8.166667
4,P104,Wheal,SCOTLAND,5,80,70,3,11.666667,6.857143


In [6]:
bat_pos_summary = batting.groupby("player_id").agg(
    avg_batting_pos=("batting_pos", "mean")
).reset_index()

# Merge into batting summary
batting_summary = pd.merge(
    batting_summary,
    bat_pos_summary,
    on="player_id",
    how="left"
)


In [7]:
performance = pd.merge(
    batting_summary,
    bowling_summary,
    on="player_id",
    how="outer"
)
performance.fillna(0, inplace=True)

In [8]:
# Ensure player_name and team exist for all players
performance = pd.merge(
    performance,
    players[["player_id", "player_name", "team"]],
    on="player_id",
    how="left"
)

In [9]:
# Merge Batting + Bowling
performance = pd.merge(
    batting_summary,
    bowling_summary,
    on="player_id",
    how="outer"
)

# Add proper player_name + team from players table
performance = pd.merge(
    performance,
    players[["player_id", "player_name", "team"]],
    on="player_id",
    how="left"
)


In [10]:
performance = pd.merge(
    performance,
    players[["player_id", "role_clean", "bowling_type"]],
    on="player_id",
    how="left"
)

performance["role_clean"] = performance["role_clean"].fillna("unknown")
performance["bowling_type"] = performance["bowling_type"].fillna("unknown")

In [11]:
def classify_dashboard_role(row):
    role = str(row["role_clean"]).lower()
    bowl_type = str(row["bowling_type"]).lower()

    # Batters + WK
    if role in ["bat", "wk"]:
        return "Batsman"

    # All Rounders
    if role == "ar":
        return "All-Rounder"

    # Bowlers
    if role == "bowl":
        spin_keywords = ["leg", "off", "orthodox", "break", "spin"]
        if any(word in bowl_type for word in spin_keywords):
            return "Spinner"
        else:
            return "Pacer"

    return "Other"

performance["dashboard_role"] = performance.apply(classify_dashboard_role, axis=1)

In [12]:
print(performance["dashboard_role"].value_counts())

dashboard_role
All-Rounder    88
Batsman        86
Pacer          70
Spinner        21
Name: count, dtype: int64


In [13]:
def classify_batting_role(row):

    if row["dashboard_role"] not in ["Batsman", "All-Rounder"]:
        return "Not a Batter"

    pos = row["avg_batting_pos"]

    if pos <= 2.5:
        return "Opener"
    elif pos <= 3.5:
        return "Anchor"
    elif pos <= 5.5:
        return "Middle Order"
    else:
        return "Finisher"

performance["batting_role"] = performance.apply(classify_batting_role, axis=1)

In [14]:
print(performance["batting_role"].value_counts())

batting_role
Not a Batter    91
Finisher        59
Middle Order    49
Opener          48
Anchor          18
Name: count, dtype: int64


In [15]:
performance["batting_impact"] = (
    performance["total_runs"] * 0.5 +
    performance["strike_rate"] * 0.3 +
    performance["boundary_pct"] * 0.2
)

In [16]:
performance["bowling_impact"] = (
    performance["total_wickets"] * 0.6 -
    performance["economy"] * 0.4
)

In [17]:
top_openers = performance[
    (performance["batting_role"] == "Opener") &
    (performance["innings_batted"] >= 4)
].sort_values("batting_impact", ascending=False).head(5)

top_openers

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
245,P77,Head,Australia,255.0,161.0,26.0,15.0,7.0,158.385093,194.0,...,NaN,NaN,Travis Head,Australia,bat,offbreak,Batsman,Opener,190.231214,NaN
0,P1,Rohit Sharma (c),India,257.0,164.0,24.0,15.0,8.0,156.707317,186.0,...,NaN,NaN,Rohit Sharma,India,bat,offbreak,Batsman,Opener,189.986903,NaN
75,P175,Gurbaz,Afghanistan,281.0,226.0,18.0,16.0,8.0,124.336283,168.0,...,NaN,NaN,Rahmanullah Gurbaz WK-,Afghanistan,wk,unknown,Batsman,Opener,189.758180,NaN
27,P125,Quinton de Kock,South Africa,243.0,173.0,21.0,13.0,9.0,140.462428,162.0,...,NaN,NaN,Quinton de Kock WK-,South Africa,wk,unknown,Batsman,Opener,176.972062,NaN
205,P37,Jos Buttler (c & wk),England,214.0,135.0,22.0,10.0,7.0,158.518519,148.0,...,NaN,NaN,Jos Buttler,England,wk,unknown,Batsman,Opener,168.387331,NaN


In [18]:
top_anchors = performance[
    (performance["batting_role"] == "Anchor") &
    (performance["innings_batted"] >= 4)
].sort_values("batting_impact", ascending=False).head(5)

top_anchors

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
105,P208,Nicholas Pooran,West Indies,228.0,156.0,15.0,17.0,7.0,146.153846,162.0,...,NaN,NaN,Nicholas Pooran WK-,West Indies,wk,offbreak,Batsman,Anchor,172.056680,NaN
257,P9,Rishabh Pant,India,171.0,134.0,19.0,6.0,8.0,127.611940,112.0,...,NaN,NaN,Rishabh Pant WK-,India,wk,unknown,Batsman,Anchor,136.882997,NaN
247,P79,Mitchell Marsh (c),Australia,125.0,107.0,13.0,5.0,7.0,116.822430,82.0,...,NaN,NaN,Mitchell Marsh,Australia,ar,fast-medium,All-Rounder,Anchor,110.666729,NaN
23,P121,Aiden Markram (c),South Africa,123.0,122.0,14.0,2.0,9.0,100.819672,68.0,...,10.0,6.9,Aiden Markram,South Africa,bat,offbreak,Batsman,Anchor,102.802812,-1.56
169,P275,Najmul Hossain Shanto (c),Bangladesh,112.0,117.0,7.0,5.0,7.0,95.726496,58.0,...,NaN,NaN,Najmul Hossain Shanto,Bangladesh,bat,offbreak,Batsman,Anchor,95.075092,NaN


In [19]:
top_middle = performance[
    (performance["batting_role"] == "Middle Order") &
    (performance["innings_batted"] >= 4)
].sort_values("batting_impact", ascending=False).head(5)

top_middle

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
208,P4,Suryakumar Yadav,India,199.0,147.0,15.0,10.0,8.0,135.374150,120.0,...,NaN,NaN,Suryakumar Yadav,India,bat,offbreak,Batsman,Middle Order,152.172546,NaN
250,P82,Stoinis,Australia,169.0,103.0,14.0,10.0,5.0,164.077670,116.0,...,17.0,8.882353,Marcus Stoinis,Australia,ar,medium,All-Rounder,Middle Order,147.451112,2.447059
28,P126,Heinrich Klaasen,South Africa,190.0,150.0,9.0,13.0,8.0,126.666667,114.0,...,NaN,NaN,Heinrich Klaasen WK-,South Africa,wk,offbreak,Batsman,Middle Order,145.000000,NaN
229,P61,Aaron Jones,United States Of America,162.0,120.0,8.0,14.0,6.0,135.000000,116.0,...,NaN,NaN,Aaron Jones,United States,bat,legbreak,Batsman,Middle Order,135.820988,NaN
200,P31,Harry Brook,England,145.0,92.0,16.0,2.0,4.0,157.608696,76.0,...,NaN,NaN,Harry Brook,England,bat,medium,Batsman,Middle Order,130.265367,NaN


In [20]:
top_allrounders = performance[
    performance["dashboard_role"] == "All-Rounder"
].sort_values("batting_impact", ascending=False).head(5)

top_allrounders

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
250,P82,Stoinis,Australia,169.0,103.0,14.0,10.0,5.0,164.077670,116.0,...,17.0,8.882353,Marcus Stoinis,Australia,ar,medium,All-Rounder,Middle Order,147.451112,2.447059
263,P96,Brandon McMullen,Scotland,140.0,82.0,13.0,8.0,3.0,170.731707,100.0,...,NaN,NaN,Brandon McMullen,Scotland,ar,medium,All-Rounder,Anchor,135.505226,NaN
218,P5,Hardik Pandya,India,144.0,95.0,11.0,9.0,6.0,151.578947,98.0,...,25.0,7.640000,Hardik Pandya,India,ar,fast-medium,All-Rounder,Finisher,131.084795,3.544000
249,P81,Maxwell,Australia,132.0,93.0,12.0,7.0,6.0,141.935484,90.0,...,12.0,8.583333,Glenn Maxwell,Australia,ar,offbreak,All-Rounder,Middle Order,122.217009,-1.633333
98,P201,Sherfane Rutherford,West Indies,121.0,82.0,5.0,9.0,6.0,147.560976,74.0,...,NaN,NaN,Sherfane Rutherford,West Indies,ar,fast-medium,All-Rounder,Finisher,116.999698,NaN


In [21]:
top_spinners = performance[
    performance["dashboard_role"] == "Spinner"
].sort_values("bowling_impact", ascending=False).head(5)

top_spinners

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
180,P286,Rishad Hossain,Bangladesh,40.0,26.0,2.0,4.0,6.0,153.846154,32.0,...,25.0,7.760000,Rishad Hossain,Bangladesh,bowl,legbreak,Spinner,Not a Batter,82.153846,5.296000
258,P90,Adam Zampa,Australia,9.0,7.0,1.0,0.0,1.0,128.571429,4.0,...,28.0,6.678571,Adam Zampa,Australia,bowl,legbreak,Spinner,Not a Batter,51.960317,5.128571
32,P132,Keshav Maharaj,South Africa,13.0,24.0,1.0,0.0,5.0,54.166667,4.0,...,28.0,6.250000,Keshav Maharaj,South Africa,bowl,orthodox,Spinner,Not a Batter,28.903846,4.100000
211,P43,Adil Rashid,England,2.0,2.0,0.0,0.0,1.0,100.000000,0.0,...,28.0,6.642857,Adil Rashid,England,bowl,legbreak,Spinner,Not a Batter,31.000000,3.342857
109,P212,Akeal Hosein,West Indies,21.0,28.0,1.0,1.0,2.0,75.000000,10.0,...,25.0,5.640000,Akeal Hosein,West Indies,bowl,orthodox,Spinner,Not a Batter,42.523810,3.144000


In [22]:
top_pacers = performance[
    performance["dashboard_role"] == "Pacer"
].sort_values("bowling_impact", ascending=False).head(5)

top_pacers

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,overs_bowled,economy,player_name,team,role_clean,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact
80,P180,Fazalhaq Farooqi,Afghanistan,6.0,5.0,1.0,0.0,3.0,120.0,4.0,...,25.333333,6.315789,Fazalhaq Farooqi,Afghanistan,bowl,fast-medium,Pacer,Not a Batter,52.333333,7.673684
31,P13,Arshdeep Singh,India,12.0,16.0,1.0,0.0,3.0,75.0,4.0,...,30.000000,7.166667,Arshdeep Singh,India,bowl,fast-medium,Pacer,Not a Batter,35.166667,7.333333
40,P14,Bumrah,India,0.0,1.0,0.0,0.0,1.0,0.0,0.0,...,29.666667,4.179775,Jasprit Bumrah,India,bowl,fast,Pacer,Not a Batter,NaN,7.328090
33,P133,Anrich Nortje,South Africa,1.0,1.0,0.0,0.0,1.0,100.0,0.0,...,35.000000,5.742857,Anrich Nortje,South Africa,bowl,fast,Pacer,Not a Batter,30.500000,6.702857
78,P179,Naveen-ul-Haq,Afghanistan,6.0,15.0,1.0,0.0,3.0,40.0,4.0,...,26.666667,6.000000,Naveen-ul-Haq,Afghanistan,bowl,fast-medium,Pacer,Not a Batter,28.333333,5.400000


In [23]:
selected_ids = set()

def pick_top(df, n, score_col):
    global selected_ids
    
    # Remove already selected players
    df = df[~df["player_id"].isin(selected_ids)]
    
    # Pick top N
    top = df.sort_values(score_col, ascending=False).head(n)
    
    # Add to selected set
    selected_ids.update(top["player_id"])
    
    return top


In [24]:
# Reset selection tracker
selected_ids = set()

# Openers (2)
final_openers = pick_top(
    performance[(performance["batting_role"]=="Opener") & (performance["innings_batted"]>=4)],
    2,
    "batting_impact"
)

# Anchor (1)
final_anchor = pick_top(
    performance[(performance["batting_role"]=="Anchor") & (performance["innings_batted"]>=4)],
    1,
    "batting_impact"
)

# Middle Order (2)
final_middle = pick_top(
    performance[(performance["batting_role"]=="Middle Order") & (performance["innings_batted"]>=4)],
    2,
    "batting_impact"
)

# All-Rounder (1)
final_allrounder = pick_top(
    performance[performance["dashboard_role"]=="All-Rounder"],
    1,
    "batting_impact"
)

# Spinners (2)
final_spinners = pick_top(
    performance[performance["dashboard_role"]=="Spinner"],
    2,
    "bowling_impact"
)

# Pacers (3)
final_pacers = pick_top(
    performance[performance["dashboard_role"]=="Pacer"],
    3,
    "bowling_impact"
)


In [25]:
final_xi = pd.concat([
    final_openers,
    final_anchor,
    final_middle,
    final_allrounder,
    final_spinners,
    final_pacers
])

final_xi["final_xi_role"] = (
    ["Opener"]*2 +
    ["Anchor"]*1 +
    ["Middle Order"]*2 +
    ["All-Rounder"]*1 +
    ["Spinner"]*2 +
    ["Pacer"]*3
)


In [26]:
# final_xi_table = final_xi[[
#     "player_name",
#     "team",
#     "final_xi_role",
#     "total_runs",
#     "strike_rate",
#     "total_wickets",
#     "economy",
#     "batting_impact",
#     "bowling_impact"
# ]].reset_index(drop=True)

# final_xi_table

In [27]:
# 1. Master Performance Summary
performance.to_csv("player_performance_summary.csv", index=False)

# 2. Role-wise Top 5 Tables
top_openers.to_csv("top5_openers.csv", index=False)
top_anchors.to_csv("top5_anchors.csv", index=False)
top_middle.to_csv("top5_middle_order.csv", index=False)

top_allrounders.to_csv("top5_allrounders.csv", index=False)
top_spinners.to_csv("top5_spinners.csv", index=False)
top_pacers.to_csv("top5_pacers.csv", index=False)

# 3. Final Best XI Table
final_xi_table.to_csv("final_best_xi.csv", index=False)

NameError: name 'final_xi_table' is not defined

In [ ]:
import os
print(os.getcwd())


In [ ]:
# Clean image URLs (remove query params)
players["image_url"] = players["image"].str.split("?").str[0]

In [ ]:
# Merge Batting + Bowling
performance = pd.merge(
    batting_summary,
    bowling_summary,
    on="player_id",
    how="outer"
)

# Add player info (name, team, role, image)
performance = pd.merge(
    performance,
    players[["player_id", "player_name", "team", "role_clean", "bowling_type", "image_url"]],
    on="player_id",
    how="left"
)

performance.fillna(0, inplace=True)

In [ ]:
# Safe Strike Rate
performance["strike_rate"] = (
    performance["total_runs"] / performance["balls_faced"]
).replace([float("inf")], 0) * 100

# Safe Economy
performance["economy"] = (
    performance["runs_conceded"] / performance["overs_bowled"]
).replace([float("inf")], 0)

# Boundary %
performance["boundary_pct"] = (
    performance["boundary_runs"] / performance["total_runs"]
).replace([float("inf")], 0) * 100

# Batting Average
performance["batting_average"] = (
    performance["total_runs"] / performance["innings_batted"]
).replace([float("inf")], 0)

# Bowling Average
performance["bowling_average"] = (
    performance["runs_conceded"] / performance["total_wickets"]
).replace([float("inf")], 0)

# Bowling Strike Rate
performance["bowling_strike_rate"] = (
    performance["balls_bowled"] / performance["total_wickets"]
).replace([float("inf")], 0)

In [ ]:
performance["batting_impact"] = (
    performance["total_runs"] * 0.5 +
    performance["strike_rate"] * 0.3 +
    performance["boundary_pct"] * 0.2
)

performance["bowling_impact"] = (
    performance["total_wickets"] * 0.6 -
    performance["economy"] * 0.4 +
    performance["bowling_strike_rate"] * -0.1
)

# Combined Impact
performance["overall_impact"] = (
    performance["batting_impact"] + performance["bowling_impact"]
)

In [ ]:
print(performance.columns)

In [28]:
del final_xi

In [29]:
# Ensure image_url exists
players["image_url"] = players["image"].str.split("?").str[0]

performance = pd.merge(
    performance,
    players[["player_id", "image_url"]],
    on="player_id",
    how="left"
)

# Averages
performance["batting_average"] = (
    performance["total_runs"] / performance["innings_batted"]
).replace([float("inf")], 0)

performance["bowling_average"] = (
    performance["runs_conceded"] / performance["total_wickets"]
).replace([float("inf")], 0)

performance["overall_impact"] = (
    performance["batting_impact"] + performance["bowling_impact"]
)

In [34]:
selected_ids = set()

final_openers = pick_top(
    performance[(performance["batting_role"]=="Opener") & (performance["innings_batted"]>=4)],
    2,
    "batting_impact"
)

final_anchor = pick_top(
    performance[(performance["batting_role"]=="Anchor") & (performance["innings_batted"]>=4)],
    1,
    "batting_impact"
)

final_middle = pick_top(
    performance[(performance["batting_role"]=="Middle Order") & (performance["innings_batted"]>=4)],
    2,
    "batting_impact"
)

final_allrounder = pick_top(
    performance[performance["dashboard_role"]=="All-Rounder"],
    1,
    "overall_impact"
)

final_spinners = pick_top(
    performance[performance["dashboard_role"]=="Spinner"],
    2,
    "bowling_impact"
)

final_pacers = pick_top(
    performance[performance["dashboard_role"]=="Pacer"],
    3,
    "bowling_impact"
)


In [35]:
final_xi = pd.concat([
    final_openers,
    final_anchor,
    final_middle,
    final_allrounder,
    final_spinners,
    final_pacers
])

final_xi["final_xi_role"] = (
    ["Opener"]*2 +
    ["Anchor"]*1 +
    ["Middle Order"]*2 +
    ["All-Rounder"]*1 +
    ["Spinner"]*2 +
    ["Pacer"]*3
)


In [36]:
final_xi = pd.concat([
    final_openers,
    final_anchor,
    final_middle,
    final_allrounder,
    final_spinners,
    final_pacers
])

final_xi["final_xi_role"] = (
    ["Opener"]*2 +
    ["Anchor"]*1 +
    ["Middle Order"]*2 +
    ["All-Rounder"]*1 +
    ["Spinner"]*2 +
    ["Pacer"]*3
)

final_xi

,player_id,player_name_x,team_x,total_runs,balls_faced,fours,sixes,innings_batted,strike_rate,boundary_runs,...,bowling_type,dashboard_role,batting_role,batting_impact,bowling_impact,image_url,batting_average,bowling_average,overall_impact,final_xi_role
245,P77,Head,Australia,255.0,161.0,26.0,15.0,7.0,158.385093,194.0,...,offbreak,Batsman,Opener,190.231214,NaN,https://static.cricbuzz.com/a/img/v1/i1/c61987...,36.428571,NaN,NaN,Opener
0,P1,Rohit Sharma (c),India,257.0,164.0,24.0,15.0,8.0,156.707317,186.0,...,offbreak,Batsman,Opener,189.986903,NaN,https://static.cricbuzz.com/a/img/v1/i1/c61651...,32.125000,NaN,NaN,Opener
105,P208,Nicholas Pooran,West Indies,228.0,156.0,15.0,17.0,7.0,146.153846,162.0,...,offbreak,Batsman,Anchor,172.056680,NaN,https://static.cricbuzz.com/a/img/v1/i1/c24472...,32.571429,NaN,NaN,Anchor
208,P4,Suryakumar Yadav,India,199.0,147.0,15.0,10.0,8.0,135.374150,120.0,...,offbreak,Batsman,Middle Order,152.172546,NaN,https://static.cricbuzz.com/a/img/v1/i1/c35248...,24.875000,NaN,NaN,Middle Order
250,P82,Stoinis,Australia,169.0,103.0,14.0,10.0,5.0,164.077670,116.0,...,medium,All-Rounder,Middle Order,147.451112,2.447059,https://static.cricbuzz.com/a/img/v1/i1/c35245...,33.800000,15.100000,149.898170,Middle Order
218,P5,Hardik Pandya,India,144.0,95.0,11.0,9.0,6.0,151.578947,98.0,...,fast-medium,All-Rounder,Finisher,131.084795,3.544000,https://static.cricbuzz.com/a/img/v1/i1/c61651...,24.000000,17.363636,134.628795,All-Rounder
180,P286,Rishad Hossain,Bangladesh,40.0,26.0,2.0,4.0,6.0,153.846154,32.0,...,legbreak,Spinner,Not a Batter,82.153846,5.296000,https://static.cricbuzz.com/a/img/v1/i1/c61645...,6.666667,13.857143,87.449846,Spinner
258,P90,Adam Zampa,Australia,9.0,7.0,1.0,0.0,1.0,128.571429,4.0,...,legbreak,Spinner,Not a Batter,51.960317,5.128571,https://static.cricbuzz.com/a/img/v1/i1/c61988...,9.000000,14.384615,57.088889,Spinner
80,P180,Fazalhaq Farooqi,Afghanistan,6.0,5.0,1.0,0.0,3.0,120.000000,4.0,...,fast-medium,Pacer,Not a Batter,52.333333,7.673684,https://static.cricbuzz.com/a/img/v1/i1/c61655...,2.000000,9.411765,60.007018,Pacer
31,P13,Arshdeep Singh,India,12.0,16.0,1.0,0.0,3.0,75.000000,4.0,...,fast-medium,Pacer,Not a Batter,35.166667,7.333333,https://static.cricbuzz.com/a/img/v1/i1/c61652...,4.000000,12.647059,42.500000,Pacer


In [37]:

final_xi.to_csv("final_best_xi.csv", index=False)
print("✅ All datasets exported successfully with images + tournament metrics.")

✅ All datasets exported successfully with images + tournament metrics.
